# Thesis tables

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
DATA = PROJECT_ROOT / 'data'

MODELS = [
    ('gemini_flash',       'Gemini 2.5 Flash',      'Gemini Flash'),
    ('gemini_flash_lite',  'Gemini 2.5 Flash Lite', 'Gemini Flash Lite'),
    ('gpt54_mini',         'GPT-5.4 Mini',          'GPT-5.4 Mini'),
    ('gpt54_nano',         'GPT-5.4 Nano',          'GPT-5.4 Nano'),
    ('grok4_fast',         'Grok 4 Fast',           'Grok 4 Fast'),
    ('grok_fast',          'Grok 3 Fast',           'Grok 3 Fast'),
]

FRAMES = {mid: pd.read_csv(DATA / f'{mid}_recipes_calculated.csv') for mid, _, _ in MODELS}
ALL = pd.concat(FRAMES.values(), ignore_index=True)
ALL['fsa_cat'] = ALL['calculated_fsa_category'].astype(str).str.lower()
ALL['category'] = ALL['category'].astype(str).str.lower()

print(f'Loaded {len(MODELS)} models, {len(ALL):,} total recipes.')


Loaded 6 models, 5,603 total recipes.


## Table 1 — Recipes retained per model

In [2]:
rows = []
total_gen, total_kept = 0, 0
for mid, full_label, _ in MODELS:
    raw = pd.read_csv(DATA / f'{mid}_recipes_raw.csv')
    kept = FRAMES[mid]
    g, k = len(raw), len(kept)
    rows.append([full_label, g, k, f'{100*k/g:.1f}'])
    total_gen += g; total_kept += k
rows.append(['Total', total_gen, total_kept, f'{100*total_kept/total_gen:.1f}'])

df_retention = pd.DataFrame(rows, columns=['Model', 'Recipes generated', 'Recipes retained', 'Retention (%)'])
print('--- Table 1: Recipes retained ---')
print(df_retention.to_string(index=False))

--- Table 1: Recipes retained ---
                Model  Recipes generated  Recipes retained Retention (%)
     Gemini 2.5 Flash               1035               884          85.4
Gemini 2.5 Flash Lite               1035               888          85.8
         GPT-5.4 Mini               1035               976          94.3
         GPT-5.4 Nano               1035               877          84.7
          Grok 4 Fast               1035               972          93.9
          Grok 3 Fast               1035              1006          97.2
                Total               6210              5603          90.2


## Table 2 — Per-model FSA/WHO scores

In [3]:
rows = []
tot = dict(n=0, fsa=0, who=0, h=0, m=0, u=0)
for mid, _, label in MODELS:
    df = FRAMES[mid]
    n = len(df)
    fsa = df['calculated_fsa_score'].mean()
    who = df['calculated_who_score'].mean()
    cat = df['calculated_fsa_category'].str.lower()
    h = (cat=='healthy').sum(); m = (cat=='moderate').sum(); u = (cat=='unhealthy').sum()
    rows.append([label, n, f'{fsa:.2f}', f'{who:.2f}', f'{100*h/n:.1f}%', f'{100*m/n:.1f}%', f'{100*u/n:.1f}%'])
    tot['n']+=n; tot['fsa']+=df['calculated_fsa_score'].sum(); tot['who']+=df['calculated_who_score'].sum()
    tot['h']+=h; tot['m']+=m; tot['u']+=u
n=tot['n']
rows.append(['All', n, f'{tot["fsa"]/n:.2f}', f'{tot["who"]/n:.2f}', f'{100*tot["h"]/n:.1f}%', f'{100*tot["m"]/n:.1f}%', f'{100*tot["u"]/n:.1f}%'])
df_per_model = pd.DataFrame(rows, columns=['Model','N','Mean FSA','Mean WHO','Healthy','Moderate','Unhealthy'])
print('--- Table 2: Per-model FSA/WHO ---')
print(df_per_model.to_string(index=False))

--- Table 2: Per-model FSA/WHO ---
            Model    N Mean FSA Mean WHO Healthy Moderate Unhealthy
     Gemini Flash  884    10.21     2.72    5.9%    12.1%     82.0%
Gemini Flash Lite  888    10.36     2.30    5.2%    11.5%     83.3%
     GPT-5.4 Mini  976    10.69     2.62    3.3%     9.8%     86.9%
     GPT-5.4 Nano  877    10.68     2.33    2.6%     9.2%     88.1%
      Grok 4 Fast  972    10.17     2.85    5.7%    13.6%     80.8%
      Grok 3 Fast 1006    10.36     2.42    3.1%     9.2%     87.7%
              All 5603    10.41     2.55    4.3%    10.9%     84.8%


## Table 3 — FSA/WHO by recipe category

In [4]:
rows = []
for cat in ['breakfast', 'dinner', 'dessert']:
    sel = ALL[ALL['category']==cat]
    n = len(sel)
    fsa = sel['calculated_fsa_score'].mean()
    who = sel['calculated_who_score'].mean()
    h = (sel['fsa_cat']=='healthy').sum(); m = (sel['fsa_cat']=='moderate').sum(); u = (sel['fsa_cat']=='unhealthy').sum()
    rows.append([cat.title(), n, f'{fsa:.2f}', f'{who:.2f}', f'{100*h/n:.1f}%', f'{100*m/n:.1f}%', f'{100*u/n:.1f}%'])
n = len(ALL)
fsa = ALL['calculated_fsa_score'].mean(); who = ALL['calculated_who_score'].mean()
h = (ALL['fsa_cat']=='healthy').sum(); m = (ALL['fsa_cat']=='moderate').sum(); u = (ALL['fsa_cat']=='unhealthy').sum()
rows.append(['All', n, f'{fsa:.2f}', f'{who:.2f}', f'{100*h/n:.1f}%', f'{100*m/n:.1f}%', f'{100*u/n:.1f}%'])
df_per_cat = pd.DataFrame(rows, columns=['Category','N','Mean FSA','Mean WHO','Healthy','Moderate','Unhealthy'])
print('--- Table 3: By recipe category ---')
print(df_per_cat.to_string(index=False))

--- Table 3: By recipe category ---
 Category    N Mean FSA Mean WHO Healthy Moderate Unhealthy
Breakfast  849    10.14     2.81    6.6%     9.8%     83.6%
   Dinner 3282    10.31     2.71    4.1%    14.5%     81.4%
  Dessert 1472    10.79     2.03    3.2%     3.6%     93.2%
      All 5603    10.41     2.55    4.3%    10.9%     84.8%


## Table 4 — FSA traffic-light bands per nutrient (all models)

In [5]:
rows = []
n = len(ALL)
for label, col in [('Fat','calculated_fsa_fat_score'),
                   ('Saturated fat','calculated_fsa_sat_fat_score'),
                   ('Sugar','calculated_fsa_sugar_score'),
                   ('Salt','calculated_fsa_salt_score')]:
    g = (ALL[col]==1).sum(); a = (ALL[col]==2).sum(); r = (ALL[col]==3).sum()
    rows.append([label, f'{100*g/n:.1f}%', f'{100*a/n:.1f}%', f'{100*r/n:.1f}%'])
df_fsa_bands = pd.DataFrame(rows, columns=['Nutrient','Green','Amber','Red'])
print('--- Table 4: FSA traffic-light bands ---')
print(df_fsa_bands.to_string(index=False))

--- Table 4: FSA traffic-light bands ---
     Nutrient Green Amber   Red
          Fat  4.8%  0.7% 94.5%
Saturated fat 14.9%  0.1% 85.1%
        Sugar 34.0%  0.2% 65.8%
         Salt 24.9%  0.7% 74.4%


## Table 5 — WHO criterion pass rates (all models)

In [6]:
rows = []
n = len(ALL)
for label, col in [('Protein (10--15% energy)',       'calculated_who_protein'),
                   ('Fat (15--30% energy)',           'calculated_who_fat'),
                   ('Saturated fat ($<$10% energy)',  'calculated_who_sat_fat'),
                   ('Carbohydrates (55--75% energy)', 'calculated_who_carbs'),
                   ('Sugar ($<$10% energy)',          'calculated_who_sugar'),
                   ('Salt ($\\leq$0.2 g/serving)',      'calculated_who_salt'),
                   ('Fibre ($\\geq$3 g/100 kcal)',      'calculated_who_fibre')]:
    p = (ALL[col]==1).sum()
    rows.append([label, f'{100*p/n:.1f}%'])
df_who_pass = pd.DataFrame(rows, columns=['Criterion','Pass rate'])
print('--- Table 5: WHO criterion pass rates ---')
print(df_who_pass.to_string(index=False))

--- Table 5: WHO criterion pass rates ---
                     Criterion Pass rate
      Protein (10--15% energy)     17.2%
          Fat (15--30% energy)     33.8%
 Saturated fat ($<$10% energy)     49.8%
Carbohydrates (55--75% energy)     10.3%
         Sugar ($<$10% energy)     51.0%
    Salt ($\leq$0.2 g/serving)     20.4%
    Fibre ($\geq$3 g/100 kcal)     72.0%


## Table 6 — FSA decomposition

In [7]:
rows = []
tot = dict(n=0, cl=0.0, rec=0.0, calc=0.0, agree_cr=0, agree_rc=0)
for mid, _, label in MODELS:
    df = FRAMES[mid]
    n = len(df)
    cl = df['claimed_fsa_score'].mean(); rec = df['claimed_calc_fsa_score'].mean(); calc = df['calculated_fsa_score'].mean()
    cl_cat = df['claimed_fsa_category'].str.lower()
    rec_cat = df['claimed_calc_fsa_category'].str.lower()
    calc_cat = df['calculated_fsa_category'].str.lower()
    agree_cr = (cl_cat==rec_cat).sum(); agree_rc = (rec_cat==calc_cat).sum()
    rows.append([label, f'{cl:.2f}', f'{rec:.2f}', f'{calc:.2f}', f'{100*agree_cr/n:.1f}%', f'{100*agree_rc/n:.1f}%'])
    tot['n']+=n; tot['cl']+=df['claimed_fsa_score'].sum(); tot['rec']+=df['claimed_calc_fsa_score'].sum(); tot['calc']+=df['calculated_fsa_score'].sum()
    tot['agree_cr']+=agree_cr; tot['agree_rc']+=agree_rc
n=tot['n']
rows.append(['All', f'{tot["cl"]/n:.2f}', f'{tot["rec"]/n:.2f}', f'{tot["calc"]/n:.2f}', f'{100*tot["agree_cr"]/n:.1f}%', f'{100*tot["agree_rc"]/n:.1f}%'])
df_fsa_decomp = pd.DataFrame(rows, columns=['Model','Claimed','Recomputed','Calculated','Cl. vs Rec.','Rec. vs Calc.'])
print('--- Table 6: FSA decomposition ---')
print(df_fsa_decomp.to_string(index=False))

--- Table 6: FSA decomposition ---
            Model Claimed Recomputed Calculated Cl. vs Rec. Rec. vs Calc.
     Gemini Flash    4.83      10.27      10.21        5.8%         82.4%
Gemini Flash Lite    6.72      10.46      10.36        9.5%         81.0%
     GPT-5.4 Mini    7.13      10.28      10.69        2.7%         87.5%
     GPT-5.4 Nano    7.73      10.47      10.68        1.7%         87.5%
      Grok 4 Fast    6.05      10.43      10.17        3.3%         86.4%
      Grok 3 Fast    5.96       9.94      10.36        3.8%         88.7%
              All    6.40      10.30      10.41        4.4%         85.7%


## Table 7 — WHO decomposition

In [8]:
rows = []
tot = dict(n=0, cl=0.0, rec=0.0, calc=0.0)
for mid, _, label in MODELS:
    df = FRAMES[mid]
    n = len(df)
    cl = df['claimed_who_score'].mean(); rec = df['claimed_calc_who_score'].mean(); calc = df['calculated_who_score'].mean()
    rows.append([label, f'{cl:.2f}', f'{rec:.2f}', f'{calc:.2f}', f'{cl-rec:+.2f}', f'{rec-calc:+.2f}'])
    tot['n']+=n; tot['cl']+=df['claimed_who_score'].sum(); tot['rec']+=df['claimed_calc_who_score'].sum(); tot['calc']+=df['calculated_who_score'].sum()
n=tot['n']
cl=tot['cl']/n; rec=tot['rec']/n; calc=tot['calc']/n
rows.append(['All', f'{cl:.2f}', f'{rec:.2f}', f'{calc:.2f}', f'{cl-rec:+.2f}', f'{rec-calc:+.2f}'])
df_who_decomp = pd.DataFrame(rows, columns=['Model','Claimed','Recomputed','Calculated','Cl.\u2212Rec.','Rec.\u2212Calc.'])
print('--- Table 7: WHO decomposition ---')
print(df_who_decomp.to_string(index=False))

--- Table 7: WHO decomposition ---
            Model Claimed Recomputed Calculated Cl.−Rec. Rec.−Calc.
     Gemini Flash    5.86       2.91       2.72    +2.95      +0.18
Gemini Flash Lite    4.79       2.17       2.30    +2.62      -0.13
     GPT-5.4 Mini    4.97       2.73       2.62    +2.25      +0.10
     GPT-5.4 Nano    4.69       2.75       2.33    +1.94      +0.42
      Grok 4 Fast    4.93       2.85       2.85    +2.08      +0.00
      Grok 3 Fast    5.12       2.49       2.42    +2.63      +0.07
              All    5.06       2.65       2.55    +2.41      +0.10


## Table 8 — Nutrient accuracy averaged across models

In [9]:
NUTRIENTS = [
    ('claimed_energy_kcal','calculated_calories.quantity','Energy (kcal/100g)'),
    ('claimed_protein_g','calculated_protein','Protein (g/100g)'),
    ('claimed_carbs_g','calculated_carbohydrate','Carbohydrates (g/100g)'),
    ('claimed_fat_g','calculated_fat','Fat (g/100g)'),
    ('claimed_sugar_g','calculated_sugar_total','Sugar (g/100g)'),
    ('claimed_fiber_g','calculated_dietary_fibre','Fibre (g/100g)'),
]

def mape(c,k):
    m = (k>0)&c.notna()&k.notna(); return 100*np.mean(np.abs((c[m]-k[m])/k[m])) if m.sum() else np.nan
def r2v(c,k):
    m = c.notna()&k.notna()
    if m.sum()<3: return np.nan
    ss_res = np.sum((c[m]-k[m])**2); ss_tot = np.sum((k[m]-k[m].mean())**2)
    return 1-ss_res/ss_tot if ss_tot>0 else np.nan
def acc(c,k):
    m = (k>0)&c.notna()&k.notna(); return 100*np.mean(np.maximum(0, 1-np.abs(c[m]-k[m])/k[m])) if m.sum() else np.nan

rows = []
for c_col, k_col, label in NUTRIENTS:
    accs, mapes, r2s = [], [], []
    for mid, _, _ in MODELS:
        df = FRAMES[mid]; w = df['total_recipe_weight_g']; ok = w>0
        c = df.loc[ok, c_col]/w[ok]*100
        k = df.loc[ok, k_col]/w[ok]*100
        accs.append(acc(c,k)); mapes.append(mape(c,k)); r2s.append(r2v(c,k))
    rows.append([label, f'{np.nanmean(accs):.1f}', f'{np.nanmean(mapes):.1f}', f'{np.nanmean(r2s):.3f}'])

# Sodium (salt → sodium conversion)
accs, mapes, r2s = [], [], []
for mid, _, _ in MODELS:
    df = FRAMES[mid]; w = df['total_recipe_weight_g']; ok = w>0
    c = (df.loc[ok,'claimed_salt_g']*1000.0/2.5)/w[ok]*100
    k = df.loc[ok,'calculated_sodium_(na)']/w[ok]*100
    accs.append(acc(c,k)); mapes.append(mape(c,k)); r2s.append(r2v(c,k))
rows.append(['Sodium (mg/100g)', f'{np.nanmean(accs):.1f}', f'{np.nanmean(mapes):.1f}', f'{np.nanmean(r2s):.3f}'])

df_avg = pd.DataFrame(rows, columns=['Nutrient','Mean Accuracy (%)','MAPE (%)','$R^2$'])
print('--- Table 8: Averaged nutrient accuracy ---')
print(df_avg.to_string(index=False))

--- Table 8: Averaged nutrient accuracy ---
              Nutrient Mean Accuracy (%) MAPE (%)  $R^2$
    Energy (kcal/100g)              76.8     24.4  0.370
      Protein (g/100g)              76.5     25.7  0.243
Carbohydrates (g/100g)              64.1     64.8  0.251
          Fat (g/100g)              70.2     32.9  0.501
        Sugar (g/100g)              64.7     39.2  0.583
        Fibre (g/100g)              65.4     43.1  0.472
      Sodium (mg/100g)              50.3    114.0 -0.037


## Table 9 — Per-model per-nutrient accuracy

In [10]:
rows = []
for mid, _, label in MODELS:
    df = FRAMES[mid]; w = df['total_recipe_weight_g']; ok = w>0
    for c_col, k_col, n_label in NUTRIENTS:
        c = df.loc[ok, c_col]/w[ok]*100
        k = df.loc[ok, k_col]/w[ok]*100
        rows.append([label, n_label, f'{acc(c,k):.1f}', f'{mape(c,k):.1f}', f'{r2v(c,k):.3f}'])
    # Sodium
    c = (df.loc[ok,'claimed_salt_g']*1000.0/2.5)/w[ok]*100
    k = df.loc[ok,'calculated_sodium_(na)']/w[ok]*100
    rows.append([label, 'Sodium (mg/100g)', f'{acc(c,k):.1f}', f'{mape(c,k):.1f}', f'{r2v(c,k):.3f}'])

df_per_model_nut = pd.DataFrame(rows, columns=['Model','Nutrient','Mean Accuracy (%)','MAPE (%)','$R^2$'])
print('--- Table 9: Per-model per-nutrient accuracy ---')
print(df_per_model_nut.to_string(index=False))

--- Table 9: Per-model per-nutrient accuracy ---
            Model               Nutrient Mean Accuracy (%) MAPE (%)  $R^2$
     Gemini Flash     Energy (kcal/100g)              87.0     13.6  0.866
     Gemini Flash       Protein (g/100g)              84.6     16.2  0.582
     Gemini Flash Carbohydrates (g/100g)              71.0     48.8  0.836
     Gemini Flash           Fat (g/100g)              78.1     23.1  0.793
     Gemini Flash         Sugar (g/100g)              73.8     27.4  0.947
     Gemini Flash         Fibre (g/100g)              73.2     37.1  0.731
     Gemini Flash       Sodium (mg/100g)              45.3    195.3  0.019
Gemini Flash Lite     Energy (kcal/100g)              66.5     38.1 -1.284
Gemini Flash Lite       Protein (g/100g)              71.2     34.6 -0.117
Gemini Flash Lite Carbohydrates (g/100g)              48.5    104.7 -1.751
Gemini Flash Lite           Fat (g/100g)              62.4     46.1 -0.517
Gemini Flash Lite         Sugar (g/100g)           

## Table 10 — Signed bias per nutrient per model (supplementary)

In [11]:
rows = []
for mid, _, label in MODELS:
    df = FRAMES[mid]; w = df['total_recipe_weight_g']; ok = w>0
    biases = []
    for c_col, k_col, n_label in NUTRIENTS:
        c = df.loc[ok, c_col]/w[ok]*100
        k = df.loc[ok, k_col]/w[ok]*100
        mask = (k>0)&c.notna()&k.notna()
        bias = 100*np.mean((c[mask]-k[mask])/k[mask]) if mask.sum() else np.nan
        biases.append(bias)
    # Sodium
    c = (df.loc[ok,'claimed_salt_g']*1000.0/2.5)/w[ok]*100
    k = df.loc[ok,'calculated_sodium_(na)']/w[ok]*100
    mask = (k>0)&c.notna()&k.notna()
    bias = 100*np.mean((c[mask]-k[mask])/k[mask]) if mask.sum() else np.nan
    biases.append(bias)
    rows.append([label] + [f'{b:+.1f}%' for b in biases])

df_bias = pd.DataFrame(rows, columns=['Model','Energy','Protein','Carbs','Fat','Sugar','Fibre','Sodium'])
print('--- Table 10: Signed bias (claimed - calculated, per 100 g) ---')
print(df_bias.to_string(index=False))

--- Table 10: Signed bias (claimed - calculated, per 100 g) ---
            Model Energy Protein  Carbs    Fat  Sugar  Fibre  Sodium
     Gemini Flash  +4.6%   +7.5% +42.4%  +6.4% -10.7% +10.5% +154.9%
Gemini Flash Lite +18.7%   +3.7% +90.4% +23.4% +43.7%  -2.3%  +93.9%
     GPT-5.4 Mini  -9.1%  -13.9% +33.3%  -7.4% -20.4% -13.9%  +17.0%
     GPT-5.4 Nano  +3.7%  +11.8% +54.1%  +8.3% -19.6% +22.9%  +50.8%
      Grok 4 Fast +22.4%   +8.2% +51.2% +25.9%  -2.5%  -2.6%  +81.5%
      Grok 3 Fast  -6.2%  -15.5% +42.0%  -5.7% -30.1% -26.7%   +0.9%


## Table 11 — Model ranking summary (supplementary)

In [12]:
rows = []
for mid, _, label in MODELS:
    df = FRAMES[mid]
    w = df['total_recipe_weight_g']; ok = w>0
    accs = []
    for c_col, k_col, _ in NUTRIENTS:
        c = df.loc[ok, c_col]/w[ok]*100; k = df.loc[ok, k_col]/w[ok]*100
        accs.append(acc(c, k))
    c = (df.loc[ok,'claimed_salt_g']*1000.0/2.5)/w[ok]*100
    k = df.loc[ok,'calculated_sodium_(na)']/w[ok]*100
    accs.append(acc(c,k))
    mean_acc = np.nanmean(accs)

    fsa_unhealthy = 100*(df['calculated_fsa_category'].str.lower()=='unhealthy').mean()
    who_calc = df['calculated_who_score'].mean()
    who_claim = df['claimed_who_score'].mean()
    rows.append([label, f'{mean_acc:.1f}%', f'{fsa_unhealthy:.1f}%', f'{who_calc:.2f}', f'{who_claim:.2f}', f'{who_claim-who_calc:+.2f}'])
df_rank = pd.DataFrame(rows, columns=['Model','Mean nutrient acc.','FSA unhealthy','WHO calc.','WHO claimed','WHO overestimation'])
print('--- Table 11: Model ranking summary ---')
print(df_rank.to_string(index=False))

--- Table 11: Model ranking summary ---
            Model Mean nutrient acc. FSA unhealthy WHO calc. WHO claimed WHO overestimation
     Gemini Flash              73.3%         82.0%      2.72        5.86              +3.14
Gemini Flash Lite              58.2%         83.3%      2.30        4.79              +2.49
     GPT-5.4 Mini              72.1%         86.9%      2.62        4.97              +2.35
     GPT-5.4 Nano              62.8%         88.1%      2.33        4.69              +2.36
      Grok 4 Fast              67.0%         80.8%      2.85        4.93              +2.08
      Grok 3 Fast              67.6%         87.7%      2.42        5.12              +2.70


## Table 12 — FSA traffic-light bands per category × nutrient

In [13]:
rows = []
for cat in ['breakfast', 'dinner', 'dessert']:
    sel = ALL[ALL['category']==cat]
    n = len(sel)
    for label, col in [('Fat',           'calculated_fsa_fat_score'),
                       ('Saturated fat', 'calculated_fsa_sat_fat_score'),
                       ('Sugar',         'calculated_fsa_sugar_score'),
                       ('Salt',          'calculated_fsa_salt_score')]:
        g = (sel[col]==1).sum(); a = (sel[col]==2).sum(); r = (sel[col]==3).sum()
        rows.append([cat.title(), label, f'{100*g/n:.1f}%', f'{100*a/n:.1f}%', f'{100*r/n:.1f}%'])
df_fsa_by_cat = pd.DataFrame(rows, columns=['Category','Nutrient','Green','Amber','Red'])
print('--- Table 12: FSA traffic-light bands per category x nutrient ---')
print(f'  N: Breakfast={ (ALL["category"]=="breakfast").sum() }, Dinner={(ALL["category"]=="dinner").sum()}, Dessert={(ALL["category"]=="dessert").sum()}')
print(df_fsa_by_cat.to_string(index=False))

--- Table 12: FSA traffic-light bands per category x nutrient ---
  N: Breakfast=849, Dinner=3282, Dessert=1472
 Category      Nutrient Green Amber   Red
Breakfast           Fat  5.3%  3.5% 91.2%
Breakfast Saturated fat 11.5%  0.4% 88.1%
Breakfast         Sugar 38.9%  1.2% 60.0%
Breakfast          Salt 33.8%  1.6% 64.5%
   Dinner           Fat  4.2%  0.3% 95.5%
   Dinner Saturated fat 20.1%  0.0% 79.9%
   Dinner         Sugar 47.6%  0.0% 52.4%
   Dinner          Salt 12.3%  0.0% 87.7%
  Dessert           Fat  5.8%  0.0% 94.2%
  Dessert Saturated fat  5.2%  0.0% 94.8%
  Dessert         Sugar  1.1%  0.1% 98.8%
  Dessert          Salt 47.8%  1.7% 50.5%


## Table 13 — WHO criterion pass rates per category

In [14]:
criteria = [
    ('Protein',  'calculated_who_protein'),
    ('Fat',      'calculated_who_fat'),
    ('Sat fat',  'calculated_who_sat_fat'),
    ('Carbs',    'calculated_who_carbs'),
    ('Sugar',    'calculated_who_sugar'),
    ('Salt',     'calculated_who_salt'),
    ('Fibre',    'calculated_who_fibre'),
]
rows = []
for cat in ['breakfast', 'dinner', 'dessert']:
    sel = ALL[ALL['category']==cat]; n = len(sel)
    rows.append([cat.title()] + [f'{100*(sel[c]==1).sum()/n:.1f}%' for _, c in criteria])
n = len(ALL)
rows.append(['All'] + [f'{100*(ALL[c]==1).sum()/n:.1f}%' for _, c in criteria])
df_who_by_cat = pd.DataFrame(rows, columns=['Category'] + [label for label, _ in criteria])
print('--- Table 13: WHO criterion pass rates per category ---')
print(df_who_by_cat.to_string(index=False))

--- Table 13: WHO criterion pass rates per category ---
 Category Protein   Fat Sat fat Carbs Sugar  Salt Fibre
Breakfast   38.8% 30.7%   46.5% 19.2% 42.9% 24.9% 77.6%
   Dinner    5.0% 45.5%   63.3%  6.6% 74.0%  6.6% 69.6%
  Dessert   32.2%  9.7%   21.5% 13.2%  4.4% 48.4% 74.1%
      All   17.2% 33.8%   49.8% 10.3% 51.0% 20.4% 72.0%


## Table 14 — FSA category agreement: claimed vs calculated, per model

In [15]:
rows = []
agree_total = 0; n_total = 0
for mid, _, label in MODELS:
    df = FRAMES[mid]; n = len(df)
    cl  = df['claimed_fsa_category'].astype(str).str.lower()
    cal = df['calculated_fsa_category'].astype(str).str.lower()
    agree = (cl == cal).sum()
    agree_total += agree; n_total += n
    rows.append([label, n, f'{100*agree/n:.1f}%'])
rows.append(['All', n_total, f'{100*agree_total/n_total:.1f}%'])
df_fsa_agree = pd.DataFrame(rows, columns=['Model','N','Cl. vs Calc.'])
print('--- Table 14: FSA category agreement: claimed vs calculated ---')
print(df_fsa_agree.to_string(index=False))

--- Table 14: FSA category agreement: claimed vs calculated ---
            Model    N Cl. vs Calc.
     Gemini Flash  884         7.2%
Gemini Flash Lite  888         8.1%
     GPT-5.4 Mini  976         3.5%
     GPT-5.4 Nano  877         4.2%
      Grok 4 Fast  972         6.6%
      Grok 3 Fast 1006         5.0%
              All 5603         5.7%


## Table 15 — WHO overestimation magnitude per model

In [16]:
rows = []
pcts_gt1 = []; pcts_gt2 = []
n_total = 0; gt1_total = 0; gt2_total = 0
for mid, _, label in MODELS:
    df = FRAMES[mid]
    diff = df['claimed_who_score'] - df['calculated_who_score']
    n = len(df)
    gt1 = (diff > 1).sum(); gt2 = (diff > 2).sum()
    p1 = 100*gt1/n; p2 = 100*gt2/n
    pcts_gt1.append(p1); pcts_gt2.append(p2)
    n_total += n; gt1_total += gt1; gt2_total += gt2
    rows.append([label, n, f'{p1:.1f}%', f'{p2:.1f}%'])
rows.append(['All', n_total, f'{100*gt1_total/n_total:.1f}%', f'{100*gt2_total/n_total:.1f}%'])
df_who_over = pd.DataFrame(rows, columns=['Model','N','> 1 point','> 2 points'])
print('--- Table 15: WHO overestimation magnitude ---')
print(df_who_over.to_string(index=False))
print(f'\nCross-model range:')
print(f'  > 1 point : min = {min(pcts_gt1):.1f}%, max = {max(pcts_gt1):.1f}%')
print(f'  > 2 points: min = {min(pcts_gt2):.1f}%, max = {max(pcts_gt2):.1f}%')

--- Table 15: WHO overestimation magnitude ---
            Model    N > 1 point > 2 points
     Gemini Flash  884     86.2%      70.9%
Gemini Flash Lite  888     80.3%      53.8%
     GPT-5.4 Mini  976     77.2%      48.0%
     GPT-5.4 Nano  877     80.7%      51.3%
      Grok 4 Fast  972     68.7%      40.5%
      Grok 3 Fast 1006     86.8%      60.1%
              All 5603     79.9%      53.9%

Cross-model range:
  > 1 point : min = 68.7%, max = 86.8%
  > 2 points: min = 40.5%, max = 70.9%
